# 05 - Full-Image Transfer Learning (Part 3)

**Goal:** Use the entire MSCTD image (not just the cropped face) for sentiment classification, with a **frozen** ImageNet-pretrained ResNet50 backbone and a custom MLP head.

**Constraint reminder:** the project brief explicitly mandates frozen backbone weights for this stage. We assert this before training.

In [1]:
# Local CPU runtime setup for VS Code/Jupyter on this machine.
import os, sys
from pathlib import Path

PROJECT_NAME = "EEEM068-Human-Sentiment-Analysis"
LOCAL_PROJECT_ROOT = Path(r"C:/Users/hp/Desktop/CNN/EEEM068-Human-Sentiment-Analysis")
ENV_PROJECT_ROOT = "EEEM068_PROJECT_ROOT"

def _is_project_root(path: Path) -> bool:
    return (path / "src" / "config.py").is_file()

def _safe_resolve(path: Path):
    try:
        return path.expanduser().resolve()
    except Exception:
        return None

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []

    env_root = os.environ.get(ENV_PROJECT_ROOT)
    if env_root:
        candidates.append(Path(env_root))

    candidates.extend([cwd, *cwd.parents, LOCAL_PROJECT_ROOT])

    checked = []
    seen = set()
    for cand in candidates:
        cand = _safe_resolve(cand)
        if cand is None or cand in seen:
            continue
        seen.add(cand)
        checked.append(cand)
        if _is_project_root(cand):
            return cand

    checked_text = "\n".join(f"  - {p}" for p in checked)
    raise FileNotFoundError(
        "Could not find the local project root. Expected src/config.py.\n"
        f"Current working directory: {cwd}\n"
        f"Checked:\n{checked_text}\n\n"
        "Open this folder in VS Code and restart the notebook kernel:\n"
        "  C:/Users/hp/Desktop/CNN/EEEM068-Human-Sentiment-Analysis\n"
        f"Or set os.environ['{ENV_PROJECT_ROOT}'] to the exact local project path."
    )

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Project root: {ROOT}")

import torch
from torch.utils.data import DataLoader
import pandas as pd

from src import config as C
from src.dataset import FullImageDataset, load_master, class_weights_from_df
from src.transforms import standard_train_transform, standard_eval_transform
from src.models import build_full_image_model
from src.train import train_classifier, default_forward
from src.evaluate import evaluate_and_save
from src.utils import seed_everything, plot_training_curves, count_parameters, format_param_count

C.ensure_dirs(); seed_everything(42)
device = torch.device('cpu')

Project root: C:\Users\hp\Desktop\CNN\EEEM068-Human-Sentiment-Analysis


## 1. Load full-image splits

In [2]:
master = load_master(C.MASTER_CSV)
train_tf = standard_train_transform(C.IMAGE_SIZE)
eval_tf  = standard_eval_transform(C.IMAGE_SIZE)

train_ds = FullImageDataset(master[master.split=='train'], transform=train_tf)
val_ds   = FullImageDataset(master[master.split=='val'],   transform=eval_tf)
test_ds  = FullImageDataset(master[master.split=='test'],  transform=eval_tf)

cfg = C.FULL_IMAGE_TRAIN
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)
len(train_ds), len(val_ds), len(test_ds)

(21367, 4572, 4431)

## 2. Build model and confirm backbone is frozen

In [3]:
model = build_full_image_model(backbone='resnet50')
trainable = count_parameters(model, True)
total = count_parameters(model, False)
print(f'Trainable / total: {format_param_count(trainable)} / {format_param_count(total)}')
# Sanity check: backbone is frozen
assert all(not p.requires_grad for p in model.feature_extractor.parameters()), \
    'Backbone parameters must be frozen for Part 3.'

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\hp/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 55.6MB/s]


Trainable / total: 1.1M / 24.6M


## 3. Train the MLP head

In [4]:
class_weights = class_weights_from_df(master[master.split=='train'])
model, history = train_classifier(
    model, train_loader, val_loader,
    cfg=cfg, save_path=C.FULL_IMAGE_CKPT,
    class_weights=class_weights,
)

Ep1/12 train:   0%|          | 0/668 [00:00<?, ?it/s]

Ep1/12  val :   0%|          | 0/143 [00:00<?, ?it/s]

[Epoch 01] train_loss=1.0973 train_acc=0.372  val_loss=1.0946 val_acc=0.383  (2137.1s)


Ep2/12 train:   0%|          | 0/668 [00:00<?, ?it/s]

Ep2/12  val :   0%|          | 0/143 [00:00<?, ?it/s]

...

Ep3/12  val :   0%|          | 0/143 [00:00<?, ?it/s]

[Epoch 03] train_loss=1.0821 train_acc=0.402  val_loss=1.0954 val_acc=0.391  (3155.7s)


Ep4/12 train:   0%|          | 0/668 [00:00<?, ?it/s]

Ep4/12  val :   0%|          | 0/143 [00:00<?, ?it/s]

[Epoch 04] train_loss=1.0723 train_acc=0.418  val_loss=1.1084 val_acc=0.373  (2182.7s)
Early stopping at epoch 4 (no improvement for 3 epochs).


In [5]:
plot_training_curves(history.to_dict(), title='Full-image (frozen ResNet50 + MLP)',
                     save_path=C.PLOTS_DIR/'full_image_training_curve.png')
plot_training_curves(history.to_dict(), title='Full-image (frozen ResNet50 + MLP)',
                     save_path=C.REPORT_FIG_DIR/'full_image_training_curve.png')

## 4. Evaluate on test

In [6]:
metrics = evaluate_and_save(
    model, test_loader,
    forward_fn=default_forward,
    name='full_image_resnet50',
)
{k:metrics[k] for k in ('accuracy','macro_f1','weighted_f1')}

predict:   0%|          | 0/139 [00:00<?, ?it/s]

{'accuracy': 0.3913337846987136,
 'macro_f1': 0.36236978202479747,
 'weighted_f1': 0.37266790665644034}

## 5. Discussion

Compared to the face model (notebook 03), the full-image model can use:
- **Scene context** (lighting, background, props) - useful when faces are absent or ambiguous.
- **Body posture** and inter-person distance.

It is therefore expected to win on the `neutral` class, where facial cues are weakest.

## 6. Observed results

Final image-level metrics live in
`outputs/metrics/full_image_resnet50_metrics.json` and the confusion
matrix in `outputs/confusion_matrices/full_image_resnet50_cm.png`.

**Headline numbers** (image-level test set, n = 4,431):

| Metric | Value |
|---|---|
| Accuracy | **0.391** |
| Macro F1 | **0.362** |
| Weighted F1 | **0.373** |

**Per-class F1.** neutral **0.492**, negative **0.345**,
positive **0.250**. The prediction we made before training is
borne out: with the backbone frozen, the head learns to lean on
*scene context* and dominates on neutral (recall **0.605**) - exactly
the class where the face branch is weakest. The cost is the positive
class, where recall collapses to **0.197**: positive scenes in MSCTD
are visually heterogeneous (varied locations, lighting, props) and
the frozen backbone has no way to specialise its features for them.

**Comparison with the face branch.** Versus Notebook 03's face model
(0.378 macro-F1), the full-image model trades:

- **+7 points of recall on neutral** (0.448 -> 0.605) - scene context
  rescues frames where the face is flat or absent;
- **-10 points of recall on positive** (0.294 -> 0.197) - a frozen
  ImageNet backbone is not tuned for smile/happiness cues at
  scene-scale.

This is exactly the *complementarity* the fusion model in Notebook 06
is designed to exploit: the two branches make different mistakes on
different classes, so a small MLP on top of their probability vectors
can recover signal that neither branch alone provides.

**Constraint compliance.** The backbone (`resnet50`) is verified
frozen in cell `build_full_image_model(...)` - only the MLP head
parameters appear in the optimiser, satisfying the Part 3
"weights of the pre-trained feature extraction backbone must remain
frozen" constraint.